# ReturnIQ: Data Cleaning, Preprocessing and Feature Engineering

## 2. Data Cleaning

This notebook prepares the collected datasets for ReturnIQ analysis.

The workflow includes:

1. Duplicate and missing-value inspection
2. Data-type and date validation
3. Dataset-specific cleaning
4. Online Retail II cleaning
5. NCH complaint cleaning
6. Relationship and ID validation
7. Aggregation of one-to-many tables
8. Integration at **one row per order item**
9. Missing-value preprocessing
10. Feature engineering
11. Final validation
12. Saving cleaned datasets for later notebooks

Important: meaningful missing values are preserved when they carry business information.

## 2.1 Clean and Validate the Geolocation Dataset

In [1]:
# Check duplicate geolocation records
%run 01_Data_Loading.ipynb
print("Original rows:", len(df_geolocation))

print("Duplicate rows:", df_geolocation.duplicated().sum())

df_geolocation[df_geolocation.duplicated()].head(10)

Customers: 99441 rows, 5 columns
Geolocation: 1000163 rows, 5 columns
Order Items: 112650 rows, 7 columns
Payments: 103886 rows, 5 columns
Reviews: 99224 rows, 7 columns
Orders: 99441 rows, 8 columns
Products: 32951 rows, 9 columns
Sellers: 3095 rows, 4 columns
Translation: 71 rows, 2 columns
Online Retail II: 525461 rows, 8 columns
Customers
Rows: 99441
Columns: 5
Column names:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Geolocation
Rows: 1000163
Columns: 5
Column names:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Order Items
Rows: 112650
Columns: 7
Column names:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Payments
Rows: 103886
Columns: 5
Column names:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Reviews
Rows: 99224
Columns: 7
Column names:
['review_id

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
72,1046,-23.545320,-46.644069,sao paulo,SP
79,1050,-23.549854,-46.643139,sao paulo,SP
80,1032,-23.540775,-46.635515,sao paulo,SP
82,1046,-23.546081,-46.644820,sao paulo,SP
86,1048,-23.547449,-46.640169,são paulo,SP


In [2]:
print("Unique ZIP codes:",
      df_geolocation["geolocation_zip_code_prefix"].nunique())

print("Total rows:",
      len(df_geolocation))

Unique ZIP codes: 19015
Total rows: 1000163


In [3]:
df_geolocation[
    df_geolocation["geolocation_zip_code_prefix"] == 1001
].head(20)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
99,1001,-23.549292,-46.633559,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
235,1001,-23.550642,-46.634410,sao paulo,SP
299,1001,-23.549698,-46.633909,sao paulo,SP
326,1001,-23.551427,-46.634074,sao paulo,SP
429,1001,-23.550498,-46.634338,sao paulo,SP
519,1001,-23.551337,-46.634027,sao paulo,SP
575,1001,-23.549779,-46.633957,são paulo,SP
583,1001,-23.551337,-46.634027,sao paulo,SP
596,1001,-23.550498,-46.634338,sao paulo,SP


In [4]:
# Make a copy before cleaning
df_geolocation_clean = df_geolocation.copy()

# Remove only exact duplicate rows
df_geolocation_clean = df_geolocation_clean.drop_duplicates()

print("Original rows:", len(df_geolocation))
print("Cleaned rows:", len(df_geolocation_clean))
print("Rows removed:", len(df_geolocation) - len(df_geolocation_clean))

Original rows: 1000163
Cleaned rows: 738332
Rows removed: 261831


In [5]:
print("Duplicates remaining:",
      df_geolocation_clean.duplicated().sum())

Duplicates remaining: 0


In [6]:
zip_counts = (
    df_geolocation_clean
    .groupby("geolocation_zip_code_prefix")
    .size()
    .sort_values(ascending=False)
)

zip_counts.head(10)

geolocation_zip_code_prefix
38400    779
35500    751
11680    727
11740    678
36400    627
38408    621
39400    620
35162    611
37200    596
35900    589
dtype: int64

In [7]:
df_geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


In [8]:
df_geolocation_clean

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000155,99965,-28.180655,-52.034367,agua santa,RS
1000156,99950,-28.072188,-52.011272,tapejara,RS
1000157,99950,-28.068864,-52.012964,tapejara,RS
1000158,99950,-28.068639,-52.010705,tapejara,RS


## 2.2 Clean and Validate Reviews

In [9]:
df_reviews_clean = df_reviews.copy()

In [10]:
print(df_reviews_clean["review_score"].value_counts().sort_index())

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [11]:
print("Invalid review scores:")

print(
    df_reviews_clean[
        ~df_reviews_clean["review_score"].isin([1, 2, 3, 4, 5])
    ][["review_id", "order_id", "review_score"]]
)

Invalid review scores:
Empty DataFrame
Columns: [review_id, order_id, review_score]
Index: []


In [12]:
df_reviews_clean["review_creation_date"] = pd.to_datetime(
    df_reviews_clean["review_creation_date"],
    errors="coerce"
)

df_reviews_clean["review_answer_timestamp"] = pd.to_datetime(
    df_reviews_clean["review_answer_timestamp"],
    errors="coerce"
)

print(df_reviews_clean[[
    "review_creation_date",
    "review_answer_timestamp"
]].dtypes)

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [13]:
print(df_reviews_clean.isnull().sum())

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


In [14]:
print("Reviews cleaned successfully!")
print("Rows:", df_reviews_clean.shape[0])
print("Columns:", df_reviews_clean.shape[1])
print("Duplicates:", df_reviews_clean.duplicated().sum())

Reviews cleaned successfully!
Rows: 99224
Columns: 7
Duplicates: 0


## 2.3 Clean and Validate Orders

In [15]:
df_orders_clean = df_orders.copy()

In [16]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    df_orders_clean[col] = pd.to_datetime(
        df_orders_clean[col],
        errors="coerce"
    )

In [17]:
print(df_orders_clean["order_status"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [18]:
print(
    df_orders_clean.groupby("order_status")[date_columns]
    .apply(lambda x: x.isnull().sum())
)

              order_purchase_timestamp  order_approved_at  \
order_status                                                
approved                             0                  0   
canceled                             0                141   
created                              0                  5   
delivered                            0                 14   
invoiced                             0                  0   
processing                           0                  0   
shipped                              0                  0   
unavailable                          0                  0   

              order_delivered_carrier_date  order_delivered_customer_date  \
order_status                                                                
approved                                 2                              2   
canceled                               550                            619   
created                                  5                              5   
deli

In [19]:
print("Orders cleaned successfully!")
print("Rows:", df_orders_clean.shape[0])
print("Columns:", df_orders_clean.shape[1])
print("Duplicates:", df_orders_clean.duplicated().sum())

Orders cleaned successfully!
Rows: 99441
Columns: 8
Duplicates: 0


In [20]:
invalid_delivery = df_orders_clean[
    df_orders_clean["order_delivered_customer_date"]
    < df_orders_clean["order_purchase_timestamp"]
]

print("Invalid delivery dates:", len(invalid_delivery))

Invalid delivery dates: 0


In [21]:
invalid_carrier = df_orders_clean[
    df_orders_clean["order_delivered_carrier_date"]
    < df_orders_clean["order_purchase_timestamp"]
]

print("Invalid carrier dates:", len(invalid_carrier))

Invalid carrier dates: 166


In [22]:
invalid_carrier[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(20)

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 13:27:00,2018-08-24 14:58:37
1111,ad133696906f6a78826daa0911b7daec,delivered,2018-06-15 15:41:22,2018-06-15 14:52:00,2018-06-22 18:09:37
1329,74e033208dc13a7b8127eb8e73d09b76,delivered,2018-05-02 10:48:44,2018-05-02 09:49:00,2018-05-07 23:06:36
1372,a6b58794fd2ba533359a76c08df576e3,delivered,2018-05-14 15:18:23,2018-05-14 13:46:00,2018-05-19 19:33:32
1864,5792e0b1c8c8a2bf53af468c9a422c88,delivered,2018-07-26 13:25:14,2018-07-26 12:42:00,2018-07-30 14:45:02
2760,c3eb293fd154223498b6551a728203e8,delivered,2018-07-19 14:06:04,2018-07-19 13:49:00,2018-07-24 19:35:36
3473,b0c2a7d04b165525254254a728c50a4e,delivered,2018-06-07 13:28:30,2018-06-07 13:22:00,2018-06-21 17:36:43
3661,2033a4586b5bec3229ebc1675a8ae092,delivered,2018-06-12 10:10:25,2018-06-12 10:09:00,2018-06-19 14:08:27
4114,08adcddad19d3acf37d1fa01cb9ded1e,delivered,2018-06-27 11:16:44,2018-06-27 10:57:00,2018-06-29 17:39:53
4159,dee6298ce7d1fb2645141ef9972157aa,shipped,2018-04-30 14:06:12,2018-04-30 12:59:00,NaT


In [23]:
print(invalid_carrier["order_status"].value_counts())

order_status
delivered    165
shipped        1
Name: count, dtype: int64


In [24]:
invalid_carrier["carrier_difference"] = (
    invalid_carrier["order_delivered_carrier_date"]
    - invalid_carrier["order_purchase_timestamp"]
)

print(invalid_carrier["carrier_difference"].describe())

count                         166
mean     -2 days +21:57:37.879519
std       13 days 06:55:25.842359
min           -172 days +18:54:07
25%             -1 days +22:58:57
50%      -1 days +23:24:00.500000
75%             -1 days +23:45:56
max             -1 days +23:59:36
Name: carrier_difference, dtype: object


In [25]:
invalid_carrier.sort_values(
    "carrier_difference"
)[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "carrier_difference"
]].head(10)

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,carrier_difference
25883,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:40:53,2018-01-26 13:35:00,-172 days +18:54:07
83321,4021cd7611d6d9ce5ffcd24817fc374f,delivered,2018-08-18 11:49:40,2018-08-14 06:22:00,-5 days +18:32:20
67844,db090a16182b263b1e896bb26c6f66cf,delivered,2018-07-13 16:14:08,2018-07-13 13:59:00,-1 days +21:44:52
65452,9711d975b961355b4b5d636857e48498,delivered,2018-06-13 15:23:50,2018-06-13 13:15:00,-1 days +21:51:10
79401,89d32b64af005178b318f76cd60f2c3c,delivered,2018-07-06 11:54:40,2018-07-06 09:48:00,-1 days +21:53:20
13969,6192897f85cb2aff6dd1fca56fddb45e,delivered,2018-07-18 13:34:22,2018-07-18 11:33:00,-1 days +21:58:38
43657,3ed559dec69ab96d9074fedaf18650eb,delivered,2018-06-15 13:20:09,2018-06-15 11:22:00,-1 days +22:01:51
4256,4e157a36ea9cf89bde6fff57a780b525,delivered,2018-08-24 14:37:50,2018-08-24 12:43:00,-1 days +22:05:10
74503,119060af27b04f5fae8b2b0e27eef7f9,delivered,2018-06-11 14:34:47,2018-06-11 12:45:00,-1 days +22:10:13
21910,e23c9dcc304042fa90d538be679fa68d,delivered,2018-06-13 11:41:23,2018-06-13 09:53:00,-1 days +22:11:37


In [26]:
invalid_carrier.sort_values(
    "carrier_difference",
    ascending=False
)[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "carrier_difference"
]].head(10)

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,carrier_difference
60308,129b63bc323d7b5b2965a143b2e72cc7,delivered,2018-05-30 14:33:24,2018-05-30 14:33:00,-1 days +23:59:36
21643,f2a150c813c1eb96fd1883c1ed8fd9e5,delivered,2018-06-19 10:14:30,2018-06-19 10:14:00,-1 days +23:59:30
45543,1cb3c680ddf904752646028ee22073a4,delivered,2018-07-27 14:08:38,2018-07-27 14:08:00,-1 days +23:59:22
85758,12e4fbad3ebed6412967a3fd5818c456,delivered,2018-06-13 11:43:43,2018-06-13 11:43:00,-1 days +23:59:17
94035,2ef2818bc1477c39bc32b093812b85e8,delivered,2018-05-28 14:33:54,2018-05-28 14:33:00,-1 days +23:59:06
17831,fcb9ec0e86e6d326a21a96c2008c8610,delivered,2018-08-16 16:28:13,2018-08-16 16:27:00,-1 days +23:58:47
4660,d454d6650d375ebc3f9667a4d2fe161c,delivered,2018-05-10 09:12:16,2018-05-10 09:11:00,-1 days +23:58:44
30771,995e90c3995e0f19a9579b90c6dcbad2,delivered,2018-07-04 14:52:16,2018-07-04 14:51:00,-1 days +23:58:44
3661,2033a4586b5bec3229ebc1675a8ae092,delivered,2018-06-12 10:10:25,2018-06-12 10:09:00,-1 days +23:58:35
39289,ffcc17a2f3331d4d25cf692060921690,delivered,2018-06-29 14:41:28,2018-06-29 14:40:00,-1 days +23:58:32


In [27]:
# Mark invalid carrier dates as missing
invalid_carrier_mask = (
    df_orders_clean["order_delivered_carrier_date"]
    < df_orders_clean["order_purchase_timestamp"]
)

df_orders_clean.loc[
    invalid_carrier_mask,
    "order_delivered_carrier_date"
] = pd.NaT

print("Invalid carrier dates fixed:", invalid_carrier_mask.sum())

print(
    "Remaining invalid carrier dates:",
    (
        df_orders_clean["order_delivered_carrier_date"]
        < df_orders_clean["order_purchase_timestamp"]
    ).sum()
)

Invalid carrier dates fixed: 166
Remaining invalid carrier dates: 0


In [28]:
print("Orders shape:", df_orders_clean.shape)

print("\nMissing values:")
print(df_orders_clean.isnull().sum())

print("\nDuplicate rows:", df_orders_clean.duplicated().sum())

Orders shape: (99441, 8)

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1949
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Duplicate rows: 0


## 2.4 Clean Products, Sellers and Category Translation

In [29]:
df_products_clean = df_products.copy()

print("Products shape:", df_products_clean.shape)

print("\nMissing values:")
print(df_products_clean.isnull().sum())

print("\nDuplicate rows:", df_products_clean.duplicated().sum())

Products shape: (32951, 9)

Missing values:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Duplicate rows: 0


In [30]:
missing_category = df_products_clean[
    df_products_clean["product_category_name"].isnull()
]

print("Rows with missing category:", len(missing_category))

print("\nMissing values in these rows:")
print(missing_category.isnull().sum())

Rows with missing category: 610

Missing values in these rows:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64


In [31]:
missing_dimensions = df_products_clean[
    df_products_clean[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isnull().any(axis=1)
]

print(missing_dimensions)

                             product_id product_category_name  \
8578   09ff539a621711667c43eba6a3bd8466                 bebes   
18851  5eb564652db742ff8f28759cd8d2652a                   NaN   

       product_name_lenght  product_description_lenght  product_photos_qty  \
8578                  60.0                       865.0                 3.0   
18851                  NaN                         NaN                 NaN   

       product_weight_g  product_length_cm  product_height_cm  \
8578                NaN                NaN                NaN   
18851               NaN                NaN                NaN   

       product_width_cm  
8578                NaN  
18851               NaN  


In [32]:
# Check whether missing-category products exist in the translation table

missing_product_ids = set(
    df_products_clean.loc[
        df_products_clean["product_category_name"].isnull(),
        "product_id"
    ]
)

translation_product_ids = set(
    df_translation["product_category_name"].dropna()
)

print("Missing-category products:", len(missing_product_ids))
print("Translation rows:", len(df_translation))

Missing-category products: 610
Translation rows: 71


In [33]:
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

missing_dimensions = df_products_clean[
    df_products_clean[physical_columns].isnull().any(axis=1)
]

print("Products with missing physical information:")
print(missing_dimensions)

Products with missing physical information:
                             product_id product_category_name  \
8578   09ff539a621711667c43eba6a3bd8466                 bebes   
18851  5eb564652db742ff8f28759cd8d2652a                   NaN   

       product_name_lenght  product_description_lenght  product_photos_qty  \
8578                  60.0                       865.0                 3.0   
18851                  NaN                         NaN                 NaN   

       product_weight_g  product_length_cm  product_height_cm  \
8578                NaN                NaN                NaN   
18851               NaN                NaN                NaN   

       product_width_cm  
8578                NaN  
18851               NaN  


In [34]:
print("Total products:", len(df_products_clean))

print(
    "Unique product IDs:",
    df_products_clean["product_id"].nunique()
)

print(
    "Duplicate product IDs:",
    df_products_clean["product_id"].duplicated().sum()
)


Total products: 32951
Unique product IDs: 32951
Duplicate product IDs: 0


In [35]:
df_sellers_clean = df_sellers.copy()

print("Sellers shape:", df_sellers_clean.shape)

print("\nMissing values:")
print(df_sellers_clean.isnull().sum())

print("\nDuplicate rows:", df_sellers_clean.duplicated().sum())

print("\nDuplicate seller IDs:",
      df_sellers_clean["seller_id"].duplicated().sum())

Sellers shape: (3095, 4)

Missing values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Duplicate rows: 0

Duplicate seller IDs: 0


In [36]:
df_translation_clean = df_translation.copy()

print("Translation shape:", df_translation_clean.shape)

print("\nMissing values:")
print(df_translation_clean.isnull().sum())

print("\nDuplicate rows:", df_translation_clean.duplicated().sum())

print("\nDuplicate Portuguese categories:",
      df_translation_clean["product_category_name"].duplicated().sum())

print("\nDuplicate English categories:",
      df_translation_clean["product_category_name_english"].duplicated().sum())

Translation shape: (71, 2)

Missing values:
product_category_name            0
product_category_name_english    0
dtype: int64

Duplicate rows: 0

Duplicate Portuguese categories: 0

Duplicate English categories: 0


## 2.5 Clean Online Retail II

Negative quantities are retained as possible return/cancellation signals. They are not automatically treated as confirmed product returns.

In [37]:
df_online_retail_clean = df_online_retail.copy()

print("Online Retail II shape:", df_online_retail_clean.shape)

print("\nMissing values:")
print(df_online_retail_clean.isnull().sum())

print("\nDuplicate rows:",
      df_online_retail_clean.duplicated().sum())

print("\nData types:")
print(df_online_retail_clean.dtypes)

Online Retail II shape: (525461, 8)

Missing values:
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

Duplicate rows: 6865

Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object


In [38]:
print("Sample transactions:")
display(df_online_retail_clean.head(10))

Sample transactions:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [39]:
print("Quantity statistics:")
print(df_online_retail_clean["Quantity"].describe())

print("\nPrice statistics:")
print(df_online_retail_clean["Price"].describe())

Quantity statistics:
count    525461.000000
mean         10.337667
std         107.424110
min       -9600.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       19152.000000
Name: Quantity, dtype: float64

Price statistics:
count    525461.000000
mean          4.688834
std         146.126914
min      -53594.360000
25%           1.250000
50%           2.100000
75%           4.210000
max       25111.090000
Name: Price, dtype: float64


In [40]:
print("Negative Quantity rows:",
      (df_online_retail_clean["Quantity"] < 0).sum())

print("Negative Price rows:",
      (df_online_retail_clean["Price"] < 0).sum())

print("\nRows with negative Quantity:")
display(
    df_online_retail_clean[
        df_online_retail_clean["Quantity"] < 0
    ].head(20)
)

print("\nRows with negative Price:")
display(
    df_online_retail_clean[
        df_online_retail_clean["Price"] < 0
    ].head(20)
)

Negative Quantity rows: 12326
Negative Price rows: 3

Rows with negative Quantity:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom



Rows with negative Price:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom


In [41]:
negative_quantity = df_online_retail_clean[
    df_online_retail_clean["Quantity"] < 0
]

print("Invoice values for negative quantities:")
print(negative_quantity["Invoice"].head(30).tolist())

print("\nInvoice prefix counts:")
print(
    negative_quantity["Invoice"]
    .astype(str)
    .str[0]
    .value_counts()
)

Invoice values for negative quantities:
['C489449', 'C489449', 'C489449', 'C489449', 'C489449', 'C489449', 'C489449', 'C489449', 'C489449', 'C489459', 'C489459', 'C489459', 'C489459', 'C489459', 'C489459', 'C489459', 'C489459', 'C489459', 'C489459', 'C489459', 'C489459', 'C489459', 489464, 489463, 489467, 'C489476', 'C489503', 'C489503', 'C489504', 'C489518']

Invoice prefix counts:
Invoice
C    10205
5     1392
4      729
Name: count, dtype: int64


In [42]:
print("Zero-price transactions:",
      (df_online_retail_clean["Price"] == 0).sum())

print("Negative-price transactions:",
      (df_online_retail_clean["Price"] < 0).sum())

print("\nNegative-price examples:")
display(
    df_online_retail_clean[
        df_online_retail_clean["Price"] < 0
    ]
)

Zero-price transactions: 3687
Negative-price transactions: 3

Negative-price examples:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom


In [43]:
zero_price = df_online_retail_clean[
    df_online_retail_clean["Price"] == 0
]

print("Zero-price rows:", len(zero_price))

print("\nSample zero-price transactions:")
display(zero_price.head(20))

print("\nZero-price descriptions:")
print(zero_price["Description"].value_counts().head(20))

print("\nZero-price invoice prefixes:")
print(
    zero_price["Invoice"]
    .astype(str)
    .str[0]
    .value_counts()
)


Zero-price rows: 3687

Sample zero-price transactions:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom



Zero-price descriptions:
Description
?                                      45
damages                                39
damaged                                38
missing                                24
OWL DOORSTOP                            8
checked                                 8
POLYESTER FILLER PAD 45x45cm            7
PICNIC BASKET WICKER LARGE              7
FLAG OF ST GEORGE CAR FLAG              6
dotcom                                  6
HEART OF WICKER SMALL                   6
AIRLINE BAG VINTAGE WORLD CHAMPION      6
given away                              6
smashed                                 5
POLYESTER FILLER PAD 40x40cm            5
SMALL POPCORN HOLDER                    5
WATERING CAN BLUE ELEPHANT              5
ENAMEL FIRE BUCKET CREAM                5
ENAMEL WASH BOWL CREAM                  5
PICNIC BASKET WICKER SMALL              5
Name: count, dtype: int64

Zero-price invoice prefixes:
Invoice
5    2579
4    1108
Name: count, dtype: int64


In [44]:
missing_customer = df_online_retail_clean[
    df_online_retail_clean["Customer ID"].isnull()
]

print("Rows with missing Customer ID:", len(missing_customer))

print("\nCountries:")
print(missing_customer["Country"].value_counts().head(15))

print("\nQuantity statistics:")
print(missing_customer["Quantity"].describe())

print("\nSample:")
display(missing_customer.head(20))

Rows with missing Customer ID: 107927

Countries:
Country
United Kingdom          106429
EIRE                       960
United Arab Emirates       114
Portugal                    77
Hong Kong                   76
Bahrain                     65
France                      62
RSA                         46
Bermuda                     34
Unspecified                 30
Sweden                      19
Lebanon                     13
Nigeria                      2
Name: count, dtype: int64

Quantity statistics:
count    107927.000000
mean          0.971045
std         128.207415
min       -9600.000000
25%           1.000000
50%           1.000000
75%           2.000000
max       10200.000000
Name: Quantity, dtype: float64

Sample:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom


In [45]:
missing_description = df_online_retail_clean[
    df_online_retail_clean["Description"].isnull()
]

print("Missing Description rows:", len(missing_description))

print("\nQuantity statistics:")
print(missing_description["Quantity"].describe())

print("\nPrice statistics:")
print(missing_description["Price"].describe())

print("\nSample:")
display(missing_description.head(20))

Missing Description rows: 2928

Quantity statistics:
count    2928.000000
mean      -21.186475
std       599.532127
min     -9600.000000
25%       -30.000000
50%        -4.500000
75%         3.000000
max      9600.000000
Name: Quantity, dtype: float64

Price statistics:
count    2928.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: Price, dtype: float64

Sample:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom
6576,489901,21098,NaN,-200,2009-12-03 09:47:00,0.0,NaN,United Kingdom
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom


In [46]:
print("\nInvoice prefix:")
print(
    missing_description["Invoice"]
    .astype(str)
    .str[0]
    .value_counts()
)

print("\nMissing Customer ID:")
print(missing_description["Customer ID"].isnull().sum())


Invoice prefix:
Invoice
5    1880
4    1048
Name: count, dtype: int64

Missing Customer ID:
2928


In [47]:
duplicate_rows = df_online_retail_clean[
    df_online_retail_clean.duplicated(keep=False)
]

print("Rows involved in duplicate groups:", len(duplicate_rows))

print("\nSample duplicate rows:")
display(
    duplicate_rows.sort_values(
        ["Invoice", "StockCode", "InvoiceDate"]
    ).head(30)
)

Rows involved in duplicate groups: 13283

Sample duplicate rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [48]:
print("Duplicate rows with negative quantity:",
      (duplicate_rows["Quantity"] < 0).sum())

print("Duplicate rows with zero price:",
      (duplicate_rows["Price"] == 0).sum())

print("Duplicate rows with positive quantity:",
      (duplicate_rows["Quantity"] > 0).sum())

Duplicate rows with negative quantity: 47
Duplicate rows with zero price: 12
Duplicate rows with positive quantity: 13236


In [49]:
# Remove exact duplicate rows
before_duplicates = len(df_online_retail_clean)

df_online_retail_clean = df_online_retail_clean.drop_duplicates()

after_duplicates = len(df_online_retail_clean)

print("Rows before removing duplicates:", before_duplicates)
print("Rows after removing duplicates:", after_duplicates)
print("Duplicate rows removed:", before_duplicates - after_duplicates)

print("Remaining duplicate rows:",
      df_online_retail_clean.duplicated().sum())

Rows before removing duplicates: 525461
Rows after removing duplicates: 518596
Duplicate rows removed: 6865
Remaining duplicate rows: 0


In [50]:
zero_price_rows = df_online_retail_clean[
    df_online_retail_clean["Price"] == 0
]

print("Zero-price rows:", len(zero_price_rows))

print("\nZero-price rows with positive quantity:",
      (zero_price_rows["Quantity"] > 0).sum())

print("Zero-price rows with negative quantity:",
      (zero_price_rows["Quantity"] < 0).sum())

print("\nTop descriptions:")
display(
    zero_price_rows["Description"]
    .value_counts()
    .head(20)
)

Zero-price rows: 3681

Zero-price rows with positive quantity: 1560
Zero-price rows with negative quantity: 2121

Top descriptions:


Description
?                                      45
damages                                39
damaged                                38
missing                                24
OWL DOORSTOP                            8
checked                                 8
POLYESTER FILLER PAD 45x45cm            7
PICNIC BASKET WICKER LARGE              7
FLAG OF ST GEORGE CAR FLAG              6
dotcom                                  6
given away                              6
smashed                                 5
POLYESTER FILLER PAD 40x40cm            5
SMALL POPCORN HOLDER                    5
WATERING CAN BLUE ELEPHANT              5
HEART OF WICKER SMALL                   5
ENAMEL FIRE BUCKET CREAM                5
ENAMEL WASH BOWL CREAM                  5
AIRLINE BAG VINTAGE WORLD CHAMPION      5
PICNIC BASKET WICKER SMALL              5
Name: count, dtype: int64

In [51]:
# Remove zero-price transactions
before_zero_price = len(df_online_retail_clean)

df_online_retail_clean = df_online_retail_clean[
    df_online_retail_clean["Price"] != 0
].copy()

after_zero_price = len(df_online_retail_clean)

print("Rows before removing zero-price records:", before_zero_price)
print("Rows after removing zero-price records:", after_zero_price)
print("Zero-price rows removed:", before_zero_price - after_zero_price)

print("\nRemaining zero-price rows:",
      (df_online_retail_clean["Price"] == 0).sum())

Rows before removing zero-price records: 518596
Rows after removing zero-price records: 514915
Zero-price rows removed: 3681

Remaining zero-price rows: 0


In [52]:
# Check negative-price records
negative_price_rows = df_online_retail_clean[
    df_online_retail_clean["Price"] < 0
]

print("Negative-price rows:", len(negative_price_rows))

display(negative_price_rows)

Negative-price rows: 3


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom


In [53]:
# Remove negative-price records
before_negative_price = len(df_online_retail_clean)

df_online_retail_clean = df_online_retail_clean[
    df_online_retail_clean["Price"] >= 0
].copy()

after_negative_price = len(df_online_retail_clean)

print("Rows before removing negative-price records:", before_negative_price)
print("Rows after removing negative-price records:", after_negative_price)
print("Negative-price rows removed:",
      before_negative_price - after_negative_price)

print("\nRemaining negative-price rows:",
      (df_online_retail_clean["Price"] < 0).sum())

Rows before removing negative-price records: 514915
Rows after removing negative-price records: 514912
Negative-price rows removed: 3

Remaining negative-price rows: 0


In [54]:
print("===== FINAL ONLINE RETAIL II CHECK =====")

print("Shape:", df_online_retail_clean.shape)

print("\nMissing values:")
print(df_online_retail_clean.isnull().sum())

print("\nDuplicate rows:",
      df_online_retail_clean.duplicated().sum())

print("\nNegative quantity:",
      (df_online_retail_clean["Quantity"] < 0).sum())

print("Zero quantity:",
      (df_online_retail_clean["Quantity"] == 0).sum())

print("Positive quantity:",
      (df_online_retail_clean["Quantity"] > 0).sum())

print("\nNegative price:",
      (df_online_retail_clean["Price"] < 0).sum())

print("Zero price:",
      (df_online_retail_clean["Price"] == 0).sum())

print("Positive price:",
      (df_online_retail_clean["Price"] > 0).sum())

print("\nData types:")
print(df_online_retail_clean.dtypes)

===== FINAL ONLINE RETAIL II CHECK =====
Shape: (514912, 8)

Missing values:
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    104180
Country             0
dtype: int64

Duplicate rows: 0

Negative quantity: 10181
Zero quantity: 0
Positive quantity: 504731

Negative price: 0
Zero price: 0
Positive price: 514912

Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object


## 2.6 Clean NCH Complaint Data

In [55]:
df_complaints = pd.read_csv('RS_Session_266_AU_2442_A.ii_.csv')

print("Shape:", df_complaints.shape)
print("\nColumns:")
print(df_complaints.columns.tolist())

print("\nMissing values:")
print(df_complaints.isnull().sum())

print("\nDuplicate rows:")
print(df_complaints.duplicated().sum())

display(df_complaints.head(15))

Shape: (12, 3)

Columns:
['Sl. No.', 'Nature of Grievances', 'Number of Grievances Received on NCH']

Missing values:
Sl. No.                                 0
Nature of Grievances                    0
Number of Grievances Received on NCH    0
dtype: int64

Duplicate rows:
0


,Sl. No.,Nature of Grievances,Number of Grievances Received on NCH
0,1,Deficiency in Services,59723
1,2,Delivery of Wrong Product,54563
2,3,Delivery of Defective / Damage Product,53285
3,4,Paid amount not refunded,50997
4,5,Non-Delivery of Product,42781
5,6,Delay in Delivery of Product,27394
6,7,Replacement / Refund not provided - As per policy,19020
7,8,Amount Debited but Not Credited to Beneficiary,15315
8,9,Product / Product Accessories Missing,15077
9,10,Account Blocked/Service Barred,5913


In [56]:
df_complaints_clean = df_complaints.copy()

# Remove the aggregate Grand Total row
df_complaints_clean = df_complaints_clean[
    df_complaints_clean["Nature of Grievances"] != "Grand Total"
].copy()

print("Original rows:", len(df_complaints))
print("Cleaned rows:", len(df_complaints_clean))

print("\nRemaining categories:")
display(df_complaints_clean)

Original rows: 12
Cleaned rows: 11

Remaining categories:


,Sl. No.,Nature of Grievances,Number of Grievances Received on NCH
0,1,Deficiency in Services,59723
1,2,Delivery of Wrong Product,54563
2,3,Delivery of Defective / Damage Product,53285
3,4,Paid amount not refunded,50997
4,5,Non-Delivery of Product,42781
5,6,Delay in Delivery of Product,27394
6,7,Replacement / Refund not provided - As per policy,19020
7,8,Amount Debited but Not Credited to Beneficiary,15315
8,9,Product / Product Accessories Missing,15077
9,10,Account Blocked/Service Barred,5913


## 2.7 Validate Relationships Between Olist Tables

In [57]:
print("CUSTOMER IDs")
print("Orders customer_id:", df_orders_clean["customer_id"].nunique())
print("Customers customer_id:", df_customers["customer_id"].nunique())

print("\nORDER IDs")
print("Orders:", df_orders_clean["order_id"].nunique())
print("Order Items:", df_order_items["order_id"].nunique())
print("Payments:", df_payments["order_id"].nunique())
print("Reviews:", df_reviews_clean["order_id"].nunique())

print("\nPRODUCT IDs")
print("Products:", df_products_clean["product_id"].nunique())
print("Order Items:", df_order_items["product_id"].nunique())

print("\nSELLER IDs")
print("Sellers:", df_sellers_clean["seller_id"].nunique())
print("Order Items:", df_order_items["seller_id"].nunique())

CUSTOMER IDs
Orders customer_id: 99441
Customers customer_id: 99441

ORDER IDs
Orders: 99441
Order Items: 98666
Payments: 99440
Reviews: 98673

PRODUCT IDs
Products: 32951
Order Items: 32951

SELLER IDs
Sellers: 3095
Order Items: 3095


In [58]:
print("===== ID MATCHING CHECK =====")

# Orders → Customers
missing_customers = set(df_orders_clean["customer_id"]) - set(df_customers["customer_id"])
print("Orders with customer_id not found in Customers:", len(missing_customers))

# Order Items → Orders
missing_order_items = set(df_order_items["order_id"]) - set(df_orders_clean["order_id"])
print("Order Items with order_id not found in Orders:", len(missing_order_items))

# Payments → Orders
missing_payments = set(df_payments["order_id"]) - set(df_orders_clean["order_id"])
print("Payments with order_id not found in Orders:", len(missing_payments))

# Reviews → Orders
missing_reviews = set(df_reviews_clean["order_id"]) - set(df_orders_clean["order_id"])
print("Reviews with order_id not found in Orders:", len(missing_reviews))

# Order Items → Products
missing_products = set(df_order_items["product_id"]) - set(df_products_clean["product_id"])
print("Order Items with product_id not found in Products:", len(missing_products))

# Order Items → Sellers
missing_sellers = set(df_order_items["seller_id"]) - set(df_sellers_clean["seller_id"])
print("Order Items with seller_id not found in Sellers:", len(missing_sellers))

===== ID MATCHING CHECK =====
Orders with customer_id not found in Customers: 0
Order Items with order_id not found in Orders: 0
Payments with order_id not found in Orders: 0
Reviews with order_id not found in Orders: 0
Order Items with product_id not found in Products: 0
Order Items with seller_id not found in Sellers: 0


In [59]:
print("===== PAYMENT RECORDS PER ORDER =====")

payment_counts = df_payments.groupby("order_id").size()

print(payment_counts.value_counts().sort_index())

print("\nMaximum payments for one order:",
      payment_counts.max())


print("\n===== REVIEW RECORDS PER ORDER =====")

review_counts = df_reviews_clean.groupby("order_id").size()

print(review_counts.value_counts().sort_index())

print("\nMaximum reviews for one order:",
      review_counts.max())

===== PAYMENT RECORDS PER ORDER =====
1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

Maximum payments for one order: 29

===== REVIEW RECORDS PER ORDER =====
1    98126
2      543
3        4
Name: count, dtype: int64

Maximum reviews for one order: 3


In [60]:
print("===== PAYMENT TYPES =====")
print(df_payments["payment_type"].value_counts())

print("\n===== PAYMENT TYPE + ORDER COUNT =====")
print(
    df_payments.groupby("payment_type")["order_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n===== PAYMENT VALUE CHECK =====")
print(df_payments["payment_value"].describe())

===== PAYMENT TYPES =====
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

===== PAYMENT TYPE + ORDER COUNT =====
payment_type
credit_card    76505
boleto         19784
voucher         3866
debit_card      1528
not_defined        3
Name: order_id, dtype: int64

===== PAYMENT VALUE CHECK =====
count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64


## 2.8 Aggregate Payments and Reviews Before Integration

In [61]:
payment_agg = (
    df_payments
    .groupby("order_id")
    .agg(
        payment_total_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        payment_max_installments=("payment_installments", "max"),
        payment_types=("payment_type", lambda x: ", ".join(sorted(x.dropna().unique())))
    )
    .reset_index()
)

print("Payment aggregated shape:", payment_agg.shape)

display(payment_agg.head())

Payment aggregated shape: (99440, 5)


,order_id,payment_total_value,payment_count,payment_max_installments,payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,credit_card


In [62]:
review_agg = (
    df_reviews_clean
    .groupby("order_id")
    .agg(
        review_score_mean=("review_score", "mean"),
        review_score_min=("review_score", "min"),
        review_score_count=("review_score", "count"),
        review_comments=("review_comment_message",
                         lambda x: " | ".join(
                             x.dropna().astype(str)
                         ))
    )
    .reset_index()
)

print("Review aggregated shape:", review_agg.shape)

display(review_agg.head())

Review aggregated shape: (98673, 5)


,order_id,review_score_mean,review_score_min,review_score_count,review_comments
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,5,1,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,4.0,4,1,
2,000229ec398224ef6ca0657da4fc703e,5.0,5,1,Chegou antes do prazo previsto e o produto sur...
3,00024acbcdf0a6daa1e931b038114c75,4.0,4,1,
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,5,1,Gostei pois veio no prazo determinado .


In [63]:
print("Payment aggregated shape:", payment_agg.shape)
display(payment_agg.head())

Payment aggregated shape: (99440, 5)


,order_id,payment_total_value,payment_count,payment_max_installments,payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,credit_card


## 2.9 Integrate Olist Data at Order-Item Grain

In [64]:
# Product information needed for ReturnIQ
products_for_merge = df_products_clean[
    [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].copy()

# Seller information needed for ReturnIQ
sellers_for_merge = df_sellers_clean[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
].copy()

print("Products for merge:", products_for_merge.shape)
print("Sellers for merge:", sellers_for_merge.shape)

Products for merge: (32951, 6)
Sellers for merge: (3095, 4)


In [65]:
order_items_products = df_order_items.merge(
    products_for_merge,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("Order Items shape:", df_order_items.shape)
print("After Products merge:", order_items_products.shape)

Order Items shape: (112650, 7)
After Products merge: (112650, 12)


In [66]:
order_items_enriched = order_items_products.merge(
    sellers_for_merge,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print("After Sellers merge:", order_items_enriched.shape)

After Sellers merge: (112650, 15)


In [67]:
orders_for_merge = df_orders_clean[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].copy()

print("Orders for merge:", orders_for_merge.shape)

Orders for merge: (99441, 8)


In [68]:
order_items_orders = order_items_enriched.merge(
    orders_for_merge,
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Before Orders merge:", order_items_enriched.shape)
print("After Orders merge:", order_items_orders.shape)

Before Orders merge: (112650, 15)
After Orders merge: (112650, 22)


In [69]:
customers_for_merge = df_customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
].copy()

print("Customers for merge:", customers_for_merge.shape)

Customers for merge: (99441, 5)


In [70]:
order_items_customers = order_items_orders.merge(
    customers_for_merge,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Before Customers merge:", order_items_orders.shape)
print("After Customers merge:", order_items_customers.shape)

Before Customers merge: (112650, 22)
After Customers merge: (112650, 26)


In [71]:
order_items_payments = order_items_customers.merge(
    payment_agg,
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Before Payments merge:", order_items_customers.shape)
print("After Payments merge:", order_items_payments.shape)

Before Payments merge: (112650, 26)
After Payments merge: (112650, 30)


In [72]:
order_items_complete = order_items_payments.merge(
    review_agg,
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Before Reviews merge:", order_items_payments.shape)
print("After Reviews merge:", order_items_complete.shape)

Before Reviews merge: (112650, 30)
After Reviews merge: (112650, 34)


In [73]:
print("Shape:", order_items_complete.shape)

print("\nColumns:")
print(order_items_complete.columns.tolist())

print("\nDuplicate rows:", order_items_complete.duplicated().sum())

print("\nMissing values:")
print(order_items_complete.isnull().sum())

Shape: (112650, 34)

Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_total_value', 'payment_count', 'payment_max_installments', 'payment_types', 'review_score_mean', 'review_score_min', 'review_score_count', 'review_comments']

Duplicate rows: 0

Missing values:
order_id                            0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freigh

In [74]:
print("Orders without payment information:",
      order_items_complete["payment_count"].isna().sum())

print("Orders without review information:",
      order_items_complete["review_score_count"].isna().sum())

print("\nOrder status distribution:")
print(order_items_complete["order_status"].value_counts())

print("\nProduct category missing:",
      order_items_complete["product_category_name"].isna().sum())

Orders without payment information: 3
Orders without review information: 942

Order status distribution:
order_status
delivered      110197
shipped          1185
canceled          542
invoiced          359
processing        357
unavailable         7
approved            3
Name: count, dtype: int64

Product category missing: 1603


In [75]:
translation_for_merge = df_translation_clean[
    [
        "product_category_name",
        "product_category_name_english"
    ]
].copy()

order_items_complete = order_items_complete.merge(
    translation_for_merge,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

print("Shape after translation:", order_items_complete.shape)

print("\nMissing English categories:",
      order_items_complete["product_category_name_english"].isna().sum())

print("\nSample:")
print(
    order_items_complete[
        ["product_category_name", "product_category_name_english"]
    ].head(10)
)

Shape after translation: (112650, 35)

Missing English categories: 1627

Sample:
   product_category_name product_category_name_english
0             cool_stuff                    cool_stuff
1               pet_shop                      pet_shop
2       moveis_decoracao               furniture_decor
3             perfumaria                     perfumery
4     ferramentas_jardim                  garden_tools
5  utilidades_domesticas                    housewares
6              telefonia                     telephony
7     ferramentas_jardim                  garden_tools
8           beleza_saude                 health_beauty
9        livros_tecnicos               books_technical


In [76]:
untranslated_categories = order_items_complete[
    order_items_complete["product_category_name"].notna() &
    order_items_complete["product_category_name_english"].isna()
]["product_category_name"].value_counts()

print(untranslated_categories)

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    15
pc_gamer                                          9
Name: count, dtype: int64


In [77]:
print("Untranslated unique categories:",
      untranslated_categories.shape[0])

print("Order items affected:",
      untranslated_categories.sum())

Untranslated unique categories: 2
Order items affected: 24


In [78]:
category_mapping = {
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_preparers",
    "pc_gamer": "gaming_pc"
}

order_items_complete["product_category_name_english"] = (
    order_items_complete["product_category_name_english"]
    .fillna(order_items_complete["product_category_name"].map(category_mapping))
)

print("Missing English categories after mapping:",
      order_items_complete["product_category_name_english"].isna().sum())

print("\nMapped categories:")
print(
    order_items_complete[
        order_items_complete["product_category_name"].isin(category_mapping.keys())
    ][
        ["product_category_name", "product_category_name_english"]
    ].drop_duplicates()
)

Missing English categories after mapping: 1603

Mapped categories:
                               product_category_name  \
3228   portateis_cozinha_e_preparadores_de_alimentos   
12976                                       pc_gamer   

         product_category_name_english  
3228   portable_kitchen_food_preparers  
12976                        gaming_pc  


In [79]:
geolocation_agg = (
    df_geolocation_clean
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean")
    )
    .reset_index()
)

print("Geolocation aggregated shape:", geolocation_agg.shape)

print("\nUnique ZIP prefixes:",
      geolocation_agg["geolocation_zip_code_prefix"].nunique())

print("\nDuplicate ZIP prefixes:",
      geolocation_agg["geolocation_zip_code_prefix"].duplicated().sum())

print("\nSample:")
print(geolocation_agg.head())

Geolocation aggregated shape: (19015, 3)

Unique ZIP prefixes: 19015

Duplicate ZIP prefixes: 0

Sample:
   geolocation_zip_code_prefix   latitude  longitude
0                         1001 -23.550227 -46.634039
1                         1002 -23.547657 -46.634991
2                         1003 -23.549000 -46.635582
3                         1004 -23.549829 -46.634792
4                         1005 -23.549547 -46.636406


In [80]:
customer_geo = geolocation_agg.rename(
    columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "latitude": "customer_latitude",
        "longitude": "customer_longitude"
    }
)

order_items_complete = order_items_complete.merge(
    customer_geo,
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

print("Shape after customer geolocation:", order_items_complete.shape)

print("\nMissing customer latitude:",
      order_items_complete["customer_latitude"].isna().sum())

print("Missing customer longitude:",
      order_items_complete["customer_longitude"].isna().sum())

Shape after customer geolocation: (112650, 37)

Missing customer latitude: 302
Missing customer longitude: 302


In [81]:
seller_geo = geolocation_agg.rename(
    columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix",
        "latitude": "seller_latitude",
        "longitude": "seller_longitude"
    }
)

order_items_complete = order_items_complete.merge(
    seller_geo,
    on="seller_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

print("Shape after seller geolocation:", order_items_complete.shape)

print("\nMissing seller latitude:",
      order_items_complete["seller_latitude"].isna().sum())

print("Missing seller longitude:",
      order_items_complete["seller_longitude"].isna().sum())

Shape after seller geolocation: (112650, 39)

Missing seller latitude: 253
Missing seller longitude: 253


In [82]:
print("Final shape:", order_items_complete.shape)

print("\nDuplicate rows:",
      order_items_complete.duplicated().sum())

print("\nTotal missing values:",
      order_items_complete.isna().sum().sum())

print("\nColumns:", len(order_items_complete.columns))

Final shape: (112650, 39)

Duplicate rows: 0

Total missing values: 12018

Columns: 39


## 2.10 Handle Missing Values and Create Availability Indicators

In [83]:
missing_summary = (
    order_items_complete.isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_summary[missing_summary > 0])

order_delivered_customer_date    2454
product_category_name            1603
product_category_name_english    1603
order_delivered_carrier_date     1381
review_score_mean                 942
review_score_min                  942
review_score_count                942
review_comments                   942
customer_latitude                 302
customer_longitude                302
seller_latitude                   253
seller_longitude                  253
product_width_cm                   18
product_length_cm                  18
product_height_cm                  18
product_weight_g                   18
order_approved_at                  15
payment_total_value                 3
payment_max_installments            3
payment_types                       3
payment_count                       3
dtype: int64


In [84]:
print("Total missing values:",
      order_items_complete.isna().sum().sum())

print("Columns with missing values:",
      (order_items_complete.isna().sum() > 0).sum())

Total missing values: 12018
Columns with missing values: 21


In [85]:
# Review availability
order_items_complete["review_available"] = (
    order_items_complete["review_score_count"].notna()
).astype(int)

# Payment availability
order_items_complete["payment_available"] = (
    order_items_complete["payment_count"].notna()
).astype(int)

print("Review available:")
print(order_items_complete["review_available"].value_counts())

print("\nPayment available:")
print(order_items_complete["payment_available"].value_counts())

Review available:
review_available
1    111708
0       942
Name: count, dtype: int64

Payment available:
payment_available
1    112647
0         3
Name: count, dtype: int64


In [86]:
order_items_complete["product_category_name"] = (
    order_items_complete["product_category_name"]
    .fillna("Unknown")
)

order_items_complete["product_category_name_english"] = (
    order_items_complete["product_category_name_english"]
    .fillna("Unknown")
)

print("Missing Portuguese category:",
      order_items_complete["product_category_name"].isna().sum())

print("Missing English category:",
      order_items_complete["product_category_name_english"].isna().sum())

Missing Portuguese category: 0
Missing English category: 0


In [87]:
measurement_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in measurement_columns:
    median_value = order_items_complete[column].median()
    
    order_items_complete[column] = (
        order_items_complete[column]
        .fillna(median_value)
    )
    
    print(column, "→ median used:", median_value)

product_weight_g → median used: 700.0
product_length_cm → median used: 25.0
product_height_cm → median used: 13.0
product_width_cm → median used: 20.0


In [88]:
print("\nMissing product measurements after filling:")

print(
    order_items_complete[measurement_columns]
    .isna()
    .sum()
)


Missing product measurements after filling:
product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64


In [89]:
payment_numeric_columns = [
    "payment_total_value",
    "payment_count",
    "payment_max_installments"
]

for column in payment_numeric_columns:
    order_items_complete[column] = (
        order_items_complete[column].fillna(0)
    )

order_items_complete["payment_types"] = (
    order_items_complete["payment_types"]
    .fillna("No payment record")
)

print("Missing payment values:")
print(
    order_items_complete[
        payment_numeric_columns + ["payment_types"]
    ].isna().sum()
)

Missing payment values:
payment_total_value         0
payment_count               0
payment_max_installments    0
payment_types               0
dtype: int64


In [90]:
order_items_complete["review_comments"] = (
    order_items_complete["review_comments"]
    .fillna("No review")
)

print("Missing review comments:",
      order_items_complete["review_comments"].isna().sum())

print("Missing review scores:",
      order_items_complete["review_score_mean"].isna().sum())

print("Missing review count:",
      order_items_complete["review_score_count"].isna().sum())

Missing review comments: 0
Missing review scores: 942
Missing review count: 942


In [91]:
order_items_complete["carrier_delivery_available"] = (
    order_items_complete["order_delivered_carrier_date"].notna()
).astype(int)

order_items_complete["customer_delivery_available"] = (
    order_items_complete["order_delivered_customer_date"].notna()
).astype(int)

print("Carrier delivery available:")
print(order_items_complete["carrier_delivery_available"].value_counts())

print("\nCustomer delivery available:")
print(order_items_complete["customer_delivery_available"].value_counts())

Carrier delivery available:
carrier_delivery_available
1    111269
0      1381
Name: count, dtype: int64

Customer delivery available:
customer_delivery_available
1    110196
0      2454
Name: count, dtype: int64


In [92]:
order_items_complete["approval_available"] = (
    order_items_complete["order_approved_at"].notna()
).astype(int)

print("Approval available:")
print(order_items_complete["approval_available"].value_counts())

print("\nMissing approval dates:",
      order_items_complete["order_approved_at"].isna().sum())

Approval available:
approval_available
1    112635
0        15
Name: count, dtype: int64

Missing approval dates: 15


In [93]:
order_items_complete["customer_location_available"] = (
    order_items_complete["customer_latitude"].notna()
).astype(int)

order_items_complete["seller_location_available"] = (
    order_items_complete["seller_latitude"].notna()
).astype(int)

print("Customer location available:")
print(order_items_complete["customer_location_available"].value_counts())

print("\nSeller location available:")
print(order_items_complete["seller_location_available"].value_counts())

print("\nMissing customer latitude:",
      order_items_complete["customer_latitude"].isna().sum())

print("Missing seller latitude:",
      order_items_complete["seller_latitude"].isna().sum())

Customer location available:
customer_location_available
1    112348
0       302
Name: count, dtype: int64

Seller location available:
seller_location_available
1    112397
0       253
Name: count, dtype: int64

Missing customer latitude: 302
Missing seller latitude: 253


In [94]:
missing_final = (
    order_items_complete.isna()
    .sum()
    .sort_values(ascending=False)
)

print("Total missing values:",
      order_items_complete.isna().sum().sum())

print("\nRemaining missing values:")
print(missing_final[missing_final > 0])

Total missing values: 7786

Remaining missing values:
order_delivered_customer_date    2454
order_delivered_carrier_date     1381
review_score_mean                 942
review_score_min                  942
review_score_count                942
customer_latitude                 302
customer_longitude                302
seller_latitude                   253
seller_longitude                  253
order_approved_at                  15
dtype: int64


In [95]:
print("Final shape:", order_items_complete.shape)

print("Duplicate rows:",
      order_items_complete.duplicated().sum())

Final shape: (112650, 46)
Duplicate rows: 0


## 2.11 Engineer Delivery, Order, Time and Logistics Features

In [96]:
order_items_complete["delivery_days"] = (
    order_items_complete["order_delivered_customer_date"]
    - order_items_complete["order_purchase_timestamp"]
).dt.days

order_items_complete["estimated_delivery_days"] = (
    order_items_complete["order_estimated_delivery_date"]
    - order_items_complete["order_purchase_timestamp"]
).dt.days

order_items_complete["delivery_delay_days"] = (
    order_items_complete["order_delivered_customer_date"]
    - order_items_complete["order_estimated_delivery_date"]
).dt.days

order_items_complete["is_delayed"] = (
    order_items_complete["delivery_delay_days"] > 0
).astype(int)

print(
    order_items_complete[
        [
            "delivery_days",
            "estimated_delivery_days",
            "delivery_delay_days",
            "is_delayed"
        ]
    ].head(10)
)

   delivery_days  estimated_delivery_days  delivery_delay_days  is_delayed
0            7.0                       15                 -9.0           0
1           16.0                       18                 -3.0           0
2            7.0                       21                -14.0           0
3            6.0                       11                 -6.0           0
4           25.0                       40                -16.0           0
5            6.0                       21                -15.0           0
6            8.0                       24                -17.0           0
7            5.0                       20                -16.0           0
8            9.0                        9                  0.0           0
9            2.0                       20                -19.0           0


In [97]:
order_items_complete["is_delayed"] = pd.NA

delivered_mask = (
    order_items_complete["order_delivered_customer_date"].notna()
)

order_items_complete.loc[delivered_mask, "is_delayed"] = (
    order_items_complete.loc[
        delivered_mask, "delivery_delay_days"
    ] > 0
).astype(int)

print(order_items_complete["is_delayed"].value_counts(dropna=False))

is_delayed
0       102931
1         7265
<NA>      2454
Name: count, dtype: int64


In [98]:
order_items_complete["item_total_value"] = (
    order_items_complete["price"]
    + order_items_complete["freight_value"]
)

order_items_complete["payment_difference"] = (
    order_items_complete["payment_total_value"]
    - order_items_complete["item_total_value"]
)

print(
    order_items_complete[
        [
            "price",
            "freight_value",
            "item_total_value",
            "payment_total_value",
            "payment_difference"
        ]
    ].head(10)
)

    price  freight_value  item_total_value  payment_total_value  \
0   58.90          13.29             72.19                72.19   
1  239.90          19.93            259.83               259.83   
2  199.00          17.87            216.87               216.87   
3   12.99          12.79             25.78                25.78   
4  199.90          18.14            218.04               218.04   
5   21.90          12.69             34.59                34.59   
6   19.90          11.85             31.75                31.75   
7  810.00          70.75            880.75               880.75   
8  145.95          11.65            157.60               157.60   
9   53.99          11.40             65.39                65.39   

   payment_difference  
0        0.000000e+00  
1        0.000000e+00  
2        0.000000e+00  
3        0.000000e+00  
4       -2.842171e-14  
5        7.105427e-15  
6        0.000000e+00  
7        0.000000e+00  
8        0.000000e+00  
9        0.000000e+00 

In [99]:
order_items_complete["payment_difference"] = (
    order_items_complete["payment_difference"].round(2)
)

print(
    order_items_complete[
        [
            "item_total_value",
            "payment_total_value",
            "payment_difference"
        ]
    ].head(10)
)

   item_total_value  payment_total_value  payment_difference
0             72.19                72.19                 0.0
1            259.83               259.83                 0.0
2            216.87               216.87                 0.0
3             25.78                25.78                 0.0
4            218.04               218.04                -0.0
5             34.59                34.59                 0.0
6             31.75                31.75                 0.0
7            880.75               880.75                 0.0
8            157.60               157.60                 0.0
9             65.39                65.39                 0.0


In [100]:
order_items_complete["order_item_count"] = (
    order_items_complete.groupby("order_id")["order_item_id"]
    .transform("count")
)

print(
    order_items_complete[
        [
            "order_id",
            "order_item_id",
            "order_item_count"
        ]
    ].head(15)
)

                            order_id  order_item_id  order_item_count
0   00010242fe8c5a6d1ba2dd792cb16214              1                 1
1   00018f77f2f0320c557190d7a144bdd3              1                 1
2   000229ec398224ef6ca0657da4fc703e              1                 1
3   00024acbcdf0a6daa1e931b038114c75              1                 1
4   00042b26cf59d7ce69dfabb4e55b4fd9              1                 1
5   00048cc3ae777c65dbb7d2a0634bc1ea              1                 1
6   00054e8431b9d7675808bcb819fb4a32              1                 1
7   000576fe39319847cbb9d288c5617fa6              1                 1
8   0005a1a1728c9d785b8e2b08b904576c              1                 1
9   0005f50442cb953dcd1d21e1fb923495              1                 1
10  00061f2a7bc09da83e415a52dc8a4af1              1                 1
11  00063b381e2406b52ad429470734ebd5              1                 1
12  0006ec9db01a64e59a68b2c340bf65a7              1                 1
13  0008288aa423d2a3

In [101]:
order_items_complete["order_total_value"] = (
    order_items_complete.groupby("order_id")["item_total_value"]
    .transform("sum")
)

print(
    order_items_complete[
        [
            "order_id",
            "item_total_value",
            "order_item_count",
            "order_total_value"
        ]
    ].head(15)
)

                            order_id  item_total_value  order_item_count  \
0   00010242fe8c5a6d1ba2dd792cb16214             72.19                 1   
1   00018f77f2f0320c557190d7a144bdd3            259.83                 1   
2   000229ec398224ef6ca0657da4fc703e            216.87                 1   
3   00024acbcdf0a6daa1e931b038114c75             25.78                 1   
4   00042b26cf59d7ce69dfabb4e55b4fd9            218.04                 1   
5   00048cc3ae777c65dbb7d2a0634bc1ea             34.59                 1   
6   00054e8431b9d7675808bcb819fb4a32             31.75                 1   
7   000576fe39319847cbb9d288c5617fa6            880.75                 1   
8   0005a1a1728c9d785b8e2b08b904576c            157.60                 1   
9   0005f50442cb953dcd1d21e1fb923495             65.39                 1   
10  00061f2a7bc09da83e415a52dc8a4af1             68.87                 1   
11  00063b381e2406b52ad429470734ebd5             57.98                 1   
12  0006ec9d

In [102]:
order_items_complete["is_multi_item_order"] = (
    order_items_complete["order_item_count"] > 1
).astype(int)

print(
    order_items_complete[
        [
            "order_id",
            "order_item_count",
            "is_multi_item_order"
        ]
    ].head(15)
)

                            order_id  order_item_count  is_multi_item_order
0   00010242fe8c5a6d1ba2dd792cb16214                 1                    0
1   00018f77f2f0320c557190d7a144bdd3                 1                    0
2   000229ec398224ef6ca0657da4fc703e                 1                    0
3   00024acbcdf0a6daa1e931b038114c75                 1                    0
4   00042b26cf59d7ce69dfabb4e55b4fd9                 1                    0
5   00048cc3ae777c65dbb7d2a0634bc1ea                 1                    0
6   00054e8431b9d7675808bcb819fb4a32                 1                    0
7   000576fe39319847cbb9d288c5617fa6                 1                    0
8   0005a1a1728c9d785b8e2b08b904576c                 1                    0
9   0005f50442cb953dcd1d21e1fb923495                 1                    0
10  00061f2a7bc09da83e415a52dc8a4af1                 1                    0
11  00063b381e2406b52ad429470734ebd5                 1                    0
12  0006ec9d

In [103]:
order_items_complete["purchase_year"] = (
    order_items_complete["order_purchase_timestamp"].dt.year
)

order_items_complete["purchase_month"] = (
    order_items_complete["order_purchase_timestamp"].dt.month
)

order_items_complete["purchase_day_of_week"] = (
    order_items_complete["order_purchase_timestamp"].dt.dayofweek
)

print(
    order_items_complete[
        [
            "order_purchase_timestamp",
            "purchase_year",
            "purchase_month",
            "purchase_day_of_week"
        ]
    ].head(10)
)

  order_purchase_timestamp  purchase_year  purchase_month  \
0      2017-09-13 08:59:02           2017               9   
1      2017-04-26 10:53:06           2017               4   
2      2018-01-14 14:33:31           2018               1   
3      2018-08-08 10:00:35           2018               8   
4      2017-02-04 13:57:51           2017               2   
5      2017-05-15 21:42:34           2017               5   
6      2017-12-10 11:53:48           2017              12   
7      2018-07-04 12:08:27           2018               7   
8      2018-03-19 18:40:33           2018               3   
9      2018-07-02 13:59:39           2018               7   

   purchase_day_of_week  
0                     2  
1                     2  
2                     6  
3                     2  
4                     5  
5                     0  
6                     6  
7                     2  
8                     0  
9                     0  


In [104]:
order_items_complete["purchase_hour"] = (
    order_items_complete["order_purchase_timestamp"].dt.hour
)

print(
    order_items_complete[
        [
            "order_purchase_timestamp",
            "purchase_hour"
        ]
    ].head(10)
)

  order_purchase_timestamp  purchase_hour
0      2017-09-13 08:59:02              8
1      2017-04-26 10:53:06             10
2      2018-01-14 14:33:31             14
3      2018-08-08 10:00:35             10
4      2017-02-04 13:57:51             13
5      2017-05-15 21:42:34             21
6      2017-12-10 11:53:48             11
7      2018-07-04 12:08:27             12
8      2018-03-19 18:40:33             18
9      2018-07-02 13:59:39             13


In [105]:
order_items_complete["is_weekend"] = (
    order_items_complete["purchase_day_of_week"] >= 5
).astype(int)

print(
    order_items_complete[
        [
            "purchase_day_of_week",
            "is_weekend"
        ]
    ].head(15)
)

    purchase_day_of_week  is_weekend
0                      2           0
1                      2           0
2                      6           1
3                      2           0
4                      5           1
5                      0           0
6                      6           1
7                      2           0
8                      0           0
9                      0           0
10                     5           1
11                     4           0
12                     1           0
13                     1           0
14                     1           0


In [106]:
order_items_complete["approval_days"] = (
    order_items_complete["order_approved_at"]
    - order_items_complete["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

print(
    order_items_complete[
        [
            "order_purchase_timestamp",
            "order_approved_at",
            "approval_days"
        ]
    ].head(10)
)

  order_purchase_timestamp   order_approved_at  approval_days
0      2017-09-13 08:59:02 2017-09-13 09:45:35       0.032326
1      2017-04-26 10:53:06 2017-04-26 11:05:13       0.008414
2      2018-01-14 14:33:31 2018-01-14 14:48:30       0.010405
3      2018-08-08 10:00:35 2018-08-08 10:10:18       0.006748
4      2017-02-04 13:57:51 2017-02-04 14:10:13       0.008588
5      2017-05-15 21:42:34 2017-05-17 03:55:27       1.258947
6      2017-12-10 11:53:48 2017-12-10 12:10:31       0.011609
7      2018-07-04 12:08:27 2018-07-05 16:35:48       1.185660
8      2018-03-19 18:40:33 2018-03-20 18:35:21       0.996389
9      2018-07-02 13:59:39 2018-07-02 14:10:56       0.007836


In [107]:
order_items_complete["shipping_preparation_days"] = (
    order_items_complete["order_delivered_carrier_date"]
    - order_items_complete["order_approved_at"]
).dt.total_seconds() / (24 * 60 * 60)

print(
    order_items_complete[
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "shipping_preparation_days"
        ]
    ].head(10)
)

    order_approved_at order_delivered_carrier_date  shipping_preparation_days
0 2017-09-13 09:45:35          2017-09-19 18:34:16                   6.367141
1 2017-04-26 11:05:13          2017-05-04 14:35:00                   8.145683
2 2018-01-14 14:48:30          2018-01-16 12:36:48                   1.908542
3 2018-08-08 10:10:18          2018-08-10 13:28:00                   2.137292
4 2017-02-04 14:10:13          2017-02-16 09:46:09                  11.816620
5 2017-05-17 03:55:27          2017-05-17 11:05:55                   0.298935
6 2017-12-10 12:10:31          2017-12-12 01:07:48                   1.539780
7 2018-07-05 16:35:48          2018-07-05 12:15:00                  -0.181111
8 2018-03-20 18:35:21          2018-03-28 00:37:42                   7.251632
9 2018-07-02 14:10:56          2018-07-03 14:25:00                   1.009769


In [108]:
order_items_complete["transit_days"] = (
    order_items_complete["order_delivered_customer_date"]
    - order_items_complete["order_delivered_carrier_date"]
).dt.total_seconds() / (24 * 60 * 60)

print(
    order_items_complete[
        [
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "transit_days"
        ]
    ].head(10)
)

  order_delivered_carrier_date order_delivered_customer_date  transit_days
0          2017-09-19 18:34:16           2017-09-20 23:43:48      1.214954
1          2017-05-04 14:35:00           2017-05-12 16:04:24      8.062083
2          2018-01-16 12:36:48           2018-01-22 13:19:16      6.029491
3          2018-08-10 13:28:00           2018-08-14 13:32:39      4.003229
4          2017-02-16 09:46:09           2017-03-01 16:42:31     13.289144
5          2017-05-17 11:05:55           2017-05-22 13:44:35      5.110185
6          2017-12-12 01:07:48           2017-12-18 22:03:38      6.872106
7          2018-07-05 12:15:00           2018-07-09 14:04:07      4.075775
8          2018-03-28 00:37:42           2018-03-29 18:17:31      1.735984
9          2018-07-03 14:25:00           2018-07-04 17:28:31      1.127442


In [109]:
order_items_complete["transit_speed"] = pd.cut(
    order_items_complete["transit_days"],
    bins=[-float("inf"), 3, 7, float("inf")],
    labels=["Fast", "Normal", "Slow"]
)

order_items_complete["transit_speed"] = (
    order_items_complete["transit_speed"]
    .astype("object")
    .fillna("Unknown")
)

print(
    order_items_complete[
        [
            "transit_days",
            "transit_speed"
        ]
    ].head(15)
)

print("\nTransit speed counts:")
print(order_items_complete["transit_speed"].value_counts())

    transit_days transit_speed
0       1.214954          Fast
1       8.062083          Slow
2       6.029491        Normal
3       4.003229        Normal
4      13.289144          Slow
5       5.110185        Normal
6       6.872106        Normal
7       4.075775        Normal
8       1.735984          Fast
9       1.127442          Fast
10      1.775972          Fast
11      7.961713          Slow
12      5.584896        Normal
13      5.813819        Normal
14      5.813819        Normal

Transit speed counts:
transit_speed
Slow       56235
Normal     34975
Fast       18799
Unknown     2641
Name: count, dtype: int64


In [110]:
order_items_complete["delivery_status"] = "Not Shipped"

order_items_complete.loc[
    order_items_complete["order_delivered_carrier_date"].notna(),
    "delivery_status"
] = "In Transit"

order_items_complete.loc[
    order_items_complete["order_delivered_customer_date"].notna(),
    "delivery_status"
] = "Delivered"

print(
    order_items_complete[
        [
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "delivery_status"
        ]
    ].head(15)
)

print("\nDelivery status counts:")
print(order_items_complete["delivery_status"].value_counts())

   order_delivered_carrier_date order_delivered_customer_date delivery_status
0           2017-09-19 18:34:16           2017-09-20 23:43:48       Delivered
1           2017-05-04 14:35:00           2017-05-12 16:04:24       Delivered
2           2018-01-16 12:36:48           2018-01-22 13:19:16       Delivered
3           2018-08-10 13:28:00           2018-08-14 13:32:39       Delivered
4           2017-02-16 09:46:09           2017-03-01 16:42:31       Delivered
5           2017-05-17 11:05:55           2017-05-22 13:44:35       Delivered
6           2017-12-12 01:07:48           2017-12-18 22:03:38       Delivered
7           2018-07-05 12:15:00           2018-07-09 14:04:07       Delivered
8           2018-03-28 00:37:42           2018-03-29 18:17:31       Delivered
9           2018-07-03 14:25:00           2018-07-04 17:28:31       Delivered
10          2018-03-27 05:26:55           2018-03-29 00:04:19       Delivered
11          2018-07-30 14:52:00           2018-08-07 13:56:52   

In [111]:
order_status_mapping = {
    "created": "Pre-Processing",
    "approved": "Pre-Processing",
    "processing": "Pre-Processing",
    "invoiced": "Pre-Processing",
    "shipped": "Shipped",
    "delivered": "Delivered",
    "canceled": "Canceled",
    "unavailable": "Unavailable"
}

order_items_complete["order_status_group"] = (
    order_items_complete["order_status"]
    .map(order_status_mapping)
    .fillna("Unknown")
)

print(
    order_items_complete[
        [
            "order_status",
            "order_status_group"
        ]
    ].drop_duplicates().sort_values("order_status")
)

print("\nOrder status group counts:")
print(order_items_complete["order_status_group"].value_counts())

      order_status order_status_group
8409      approved     Pre-Processing
84        canceled           Canceled
0        delivered          Delivered
590       invoiced     Pre-Processing
1845    processing     Pre-Processing
79         shipped            Shipped
11558  unavailable        Unavailable

Order status group counts:
order_status_group
Delivered         110197
Shipped             1185
Pre-Processing       719
Canceled             542
Unavailable            7
Name: count, dtype: int64


In [112]:
import numpy as np

# Convert latitude and longitude to radians
seller_lat = np.radians(order_items_complete["seller_latitude"])
seller_lon = np.radians(order_items_complete["seller_longitude"])

customer_lat = np.radians(order_items_complete["customer_latitude"])
customer_lon = np.radians(order_items_complete["customer_longitude"])

# Differences
dlat = customer_lat - seller_lat
dlon = customer_lon - seller_lon

# Haversine formula
a = (
    np.sin(dlat / 2) ** 2
    + np.cos(seller_lat)
    * np.cos(customer_lat)
    * np.sin(dlon / 2) ** 2
)

c = 2 * np.arcsin(np.sqrt(a))

# Earth's radius in kilometers
earth_radius_km = 6371

order_items_complete["seller_customer_distance_km"] = (
    earth_radius_km * c
)

print(
    order_items_complete[
        [
            "seller_city",
            "customer_city",
            "seller_customer_distance_km"
        ]
    ].head(10)
)

print("\nMissing distance values:")
print(
    order_items_complete["seller_customer_distance_km"].isna().sum()
)

           seller_city          customer_city  seller_customer_distance_km
0        volta redonda  campos dos goytacazes                   301.419473
1            sao paulo        santa fe do sul                   585.068784
2        borda da mata          para de minas                   312.386590
3               franca                atibaia                   295.481182
4               loanda        varzea paulista                   646.149244
5       ribeirao preto                uberaba                   161.852749
6            sao paulo             guararapes                   484.837176
7  presidente prudente           praia grande                   548.135367
8            sao paulo                 santos                    53.219830
9            sao paulo                jandira                    33.274532

Missing distance values:
554


In [113]:
order_items_complete["distance_category"] = pd.cut(
    order_items_complete["seller_customer_distance_km"],
    bins=[-float("inf"), 100, 300, 600, float("inf")],
    labels=["Near", "Medium", "Far", "Very Far"]
)

order_items_complete["distance_category"] = (
    order_items_complete["distance_category"]
    .astype("object")
    .fillna("Unknown")
)

print(
    order_items_complete[
        [
            "seller_customer_distance_km",
            "distance_category"
        ]
    ].head(15)
)

print("\nDistance category counts:")
print(
    order_items_complete["distance_category"]
    .value_counts()
)

    seller_customer_distance_km distance_category
0                    301.419473               Far
1                    585.068784               Far
2                    312.386590               Far
3                    295.481182            Medium
4                    646.149244          Very Far
5                    161.852749            Medium
6                    484.837176               Far
7                    548.135367               Far
8                     53.219830              Near
9                     33.274532              Near
10                   154.117733            Medium
11                   318.182441               Far
12                   566.305580               Far
13                   396.837954               Far
14                   396.837954               Far

Distance category counts:
distance_category
Very Far    38564
Far         36958
Near        20993
Medium      15581
Unknown       554
Name: count, dtype: int64


In [114]:
order_items_complete["freight_to_price_ratio"] = (
    order_items_complete["freight_value"]
    / order_items_complete["price"]
)

print(
    order_items_complete[
        [
            "price",
            "freight_value",
            "freight_to_price_ratio"
        ]
    ].head(15)
)

print("\nMissing ratio values:")
print(
    order_items_complete["freight_to_price_ratio"].isna().sum()
)

print("\nInfinite ratio values:")
print(
    np.isinf(
        order_items_complete["freight_to_price_ratio"]
    ).sum()
)

     price  freight_value  freight_to_price_ratio
0    58.90          13.29                0.225637
1   239.90          19.93                0.083076
2   199.00          17.87                0.089799
3    12.99          12.79                0.984604
4   199.90          18.14                0.090745
5    21.90          12.69                0.579452
6    19.90          11.85                0.595477
7   810.00          70.75                0.087346
8   145.95          11.65                0.079822
9    53.99          11.40                0.211150
10   59.99           8.88                0.148025
11   45.00          12.98                0.288444
12   74.00          23.32                0.315135
13   49.90          13.37                0.267936
14   49.90          13.37                0.267936

Missing ratio values:
0

Infinite ratio values:
0


In [115]:
order_items_complete["high_freight_ratio"] = (
    order_items_complete["freight_to_price_ratio"] >= 0.50
).astype(int)

print(
    order_items_complete[
        [
            "price",
            "freight_value",
            "freight_to_price_ratio",
            "high_freight_ratio"
        ]
    ].head(15)
)

print("\nHigh freight ratio counts:")
print(
    order_items_complete["high_freight_ratio"]
    .value_counts()
)

     price  freight_value  freight_to_price_ratio  high_freight_ratio
0    58.90          13.29                0.225637                   0
1   239.90          19.93                0.083076                   0
2   199.00          17.87                0.089799                   0
3    12.99          12.79                0.984604                   1
4   199.90          18.14                0.090745                   0
5    21.90          12.69                0.579452                   1
6    19.90          11.85                0.595477                   1
7   810.00          70.75                0.087346                   0
8   145.95          11.65                0.079822                   0
9    53.99          11.40                0.211150                   0
10   59.99           8.88                0.148025                   0
11   45.00          12.98                0.288444                   0
12   74.00          23.32                0.315135                   0
13   49.90          

In [116]:
order_items_complete["product_volume_cm3"] = (
    order_items_complete["product_length_cm"]
    * order_items_complete["product_height_cm"]
    * order_items_complete["product_width_cm"]
)

print(
    order_items_complete[
        [
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
            "product_volume_cm3"
        ]
    ].head(15)
)

print("\nMissing product volume values:")
print(
    order_items_complete["product_volume_cm3"].isna().sum()
)

print("\nProduct volume statistics:")
print(
    order_items_complete["product_volume_cm3"].describe()
)

    product_length_cm  product_height_cm  product_width_cm  product_volume_cm3
0                28.0                9.0              14.0              3528.0
1                50.0               30.0              40.0             60000.0
2                33.0               13.0              33.0             14157.0
3                16.0               10.0              15.0              2400.0
4                35.0               40.0              30.0             42000.0
5                24.0                8.0              15.0              2880.0
6                27.0                5.0              20.0              2700.0
7                35.0               75.0              45.0            118125.0
8                30.0               12.0              16.0              5760.0
9                29.0                3.0              21.0              1827.0
10               35.0               25.0              20.0             17500.0
11               16.0               16.0            

## 2.12 Final Validation and Correction of Invalid Derived Durations

In [117]:
print("========== FINAL DATASET VERIFICATION ==========\n")

# 1. Shape
print("Dataset shape:", order_items_complete.shape)

# 2. Duplicate rows
print("\nDuplicate rows:")
print(order_items_complete.duplicated().sum())

# 3. Total missing values
print("\nTotal missing values:")
print(order_items_complete.isna().sum().sum())

# 4. Columns with missing values
print("\nColumns with missing values:")
missing_columns = order_items_complete.isna().sum()
print(missing_columns[missing_columns > 0])

# 5. Data types
print("\nData types:")
print(order_items_complete.dtypes)

# 6. Numeric summary
print("\nNumeric columns summary:")
print(order_items_complete.describe())

# 7. Order status
print("\nOrder status:")
print(order_items_complete["order_status"].value_counts())

# 8. Delivery status
print("\nDelivery status:")
print(order_items_complete["delivery_status"].value_counts())

# 9. Distance category
print("\nDistance category:")
print(order_items_complete["distance_category"].value_counts())

# 10. High freight ratio
print("\nHigh freight ratio:")
print(order_items_complete["high_freight_ratio"].value_counts())

print("\n========== VERIFICATION COMPLETE ==========")

========== FINAL DATASET VERIFICATION ==========

Dataset shape: (112650, 71)

Duplicate rows:
0

Total missing values:
19754

Columns with missing values:
order_approved_at                  15
order_delivered_carrier_date     1381
order_delivered_customer_date    2454
review_score_mean                 942
review_score_min                  942
review_score_count                942
customer_latitude                 302
customer_longitude                302
seller_latitude                   253
seller_longitude                  253
delivery_days                    2454
delivery_delay_days              2454
is_delayed                       2454
approval_days                      15
shipping_preparation_days        1396
transit_days                     2641
seller_customer_distance_km       554
dtype: int64

Data types:
order_id                           str
order_item_id                    int64
product_id                         str
seller_id                          str
shipping_limit_d

In [118]:
print("Negative shipping preparation days:")
print(
    (order_items_complete["shipping_preparation_days"] < 0).sum()
)

print("\nNegative transit days:")
print(
    (order_items_complete["transit_days"] < 0).sum()
)

print("\nNegative shipping preparation examples:")
print(
    order_items_complete.loc[
        order_items_complete["shipping_preparation_days"] < 0,
        [
            "order_id",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "shipping_preparation_days"
        ]
    ].head(10)
)

print("\nNegative transit examples:")
print(
    order_items_complete.loc[
        order_items_complete["transit_days"] < 0,
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "transit_days"
        ]
    ].head(10)
)

Negative shipping preparation days:
1364

Negative transit days:
50

Negative shipping preparation examples:
                              order_id order_purchase_timestamp  \
7     000576fe39319847cbb9d288c5617fa6      2018-07-04 12:08:27   
56    002175704e8b209f61b9ad5cfd92b60e      2018-04-22 12:13:25   
67    002834535f7a609a5c68266f173fa59e      2018-07-23 17:26:00   
118   004345d16a1ab2c21962992c721c8643      2018-07-04 06:45:05   
283   00b4c651133c06fb175123a492f1cac3      2018-08-17 12:56:21   
526   0132f39e8ad9b42ad2d41c847f44c0ed      2018-07-05 14:49:56   
529   0133a47b53fee66218790a8d7dcab993      2018-04-20 19:05:03   
655   0184d4ddb259e1a4cfc2871888cf97b8      2017-09-01 20:04:28   
804   01d80471e02a3ba49c8e1d0abc2c1a64      2017-05-03 19:35:37   
1293  030475e22d114f70be1a265fee326192      2018-04-23 16:06:44   

       order_approved_at order_delivered_carrier_date  \
7    2018-07-05 16:35:48          2018-07-05 12:15:00   
56   2018-04-24 17:24:48          2018-

In [119]:
# Count invalid calculated durations
invalid_shipping = (
    order_items_complete["shipping_preparation_days"] < 0
)

invalid_transit = (
    order_items_complete["transit_days"] < 0
)

print("Invalid shipping preparation durations:",
      invalid_shipping.sum())

print("Invalid transit durations:",
      invalid_transit.sum())

# Set invalid derived durations to NaN
order_items_complete.loc[
    invalid_shipping,
    "shipping_preparation_days"
] = np.nan

order_items_complete.loc[
    invalid_transit,
    "transit_days"
] = np.nan

# Verify
print("\nNegative shipping preparation durations remaining:",
      (order_items_complete["shipping_preparation_days"] < 0).sum())

print("Negative transit durations remaining:",
      (order_items_complete["transit_days"] < 0).sum())

print("\nMissing values after correction:")
print(
    order_items_complete[
        [
            "shipping_preparation_days",
            "transit_days"
        ]
    ].isna().sum()
)

Invalid shipping preparation durations: 1364
Invalid transit durations: 50

Negative shipping preparation durations remaining: 0
Negative transit durations remaining: 0

Missing values after correction:
shipping_preparation_days    2760
transit_days                 2691
dtype: int64


In [120]:
# Recalculate transit speed using the corrected transit_days
order_items_complete["transit_speed"] = pd.cut(
    order_items_complete["transit_days"],
    bins=[-float("inf"), 3, 7, float("inf")],
    labels=["Fast", "Normal", "Slow"]
)

# Convert to object and label unavailable values
order_items_complete["transit_speed"] = (
    order_items_complete["transit_speed"]
    .astype("object")
    .fillna("Unknown")
)

print(
    order_items_complete[
        [
            "transit_days",
            "transit_speed"
        ]
    ].head(15)
)

print("\nTransit speed counts:")
print(
    order_items_complete["transit_speed"].value_counts()
)

    transit_days transit_speed
0       1.214954          Fast
1       8.062083          Slow
2       6.029491        Normal
3       4.003229        Normal
4      13.289144          Slow
5       5.110185        Normal
6       6.872106        Normal
7       4.075775        Normal
8       1.735984          Fast
9       1.127442          Fast
10      1.775972          Fast
11      7.961713          Slow
12      5.584896        Normal
13      5.813819        Normal
14      5.813819        Normal

Transit speed counts:
transit_speed
Slow       56235
Normal     34975
Fast       18749
Unknown     2691
Name: count, dtype: int64


In [121]:
print("Total columns:", len(order_items_complete.columns))

print("\n========== ALL FEATURES ==========\n")

for i, column in enumerate(order_items_complete.columns, start=1):
    print(f"{i:02d}. {column}")

print("\n========== CONSTANT COLUMNS ==========\n")

constant_columns = [
    column
    for column in order_items_complete.columns
    if order_items_complete[column].nunique(dropna=False) <= 1
]

print(constant_columns)

print("\n========== DUPLICATE COLUMNS ==========\n")

duplicate_columns = []

columns = order_items_complete.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        if order_items_complete[columns[i]].equals(
            order_items_complete[columns[j]]
        ):
            duplicate_columns.append(
                (columns[i], columns[j])
            )

print(duplicate_columns)

Total columns: 71

========== ALL FEATURES ==========

01. order_id
02. order_item_id
03. product_id
04. seller_id
05. shipping_limit_date
06. price
07. freight_value
08. product_category_name
09. product_weight_g
10. product_length_cm
11. product_height_cm
12. product_width_cm
13. seller_zip_code_prefix
14. seller_city
15. seller_state
16. customer_id
17. order_status
18. order_purchase_timestamp
19. order_approved_at
20. order_delivered_carrier_date
21. order_delivered_customer_date
22. order_estimated_delivery_date
23. customer_unique_id
24. customer_zip_code_prefix
25. customer_city
26. customer_state
27. payment_total_value
28. payment_count
29. payment_max_installments
30. payment_types
31. review_score_mean
32. review_score_min
33. review_score_count
34. review_comments
35. product_category_name_english
36. customer_latitude
37. customer_longitude
38. seller_latitude
39. seller_longitude
40. review_available
41. payment_available
42. carrier_delivery_available
43. customer_deliv

## 2.13 Save Cleaned Datasets

The cleaned source datasets are saved for reuse. The integrated ReturnIQ analytical dataset is also saved
so that the EDA and visualization notebooks can run independently without repeating the entire cleaning process.

In [122]:
import os

os.makedirs("data/cleaned", exist_ok=True)

print("Cleaned data folder ready!")

Cleaned data folder ready!


In [123]:
# ============================================
# SAVE RETURNIQ CLEANED DATASETS
# ============================================

cleaned_datasets = {
    "customers_cleaned.csv": df_customers,
    "geolocation_cleaned.csv": df_geolocation_clean,
    "order_items_cleaned.csv": df_order_items,
    "payments_cleaned.csv": df_payments,
    "reviews_cleaned.csv": df_reviews_clean,
    "orders_cleaned.csv": df_orders_clean,
    "products_cleaned.csv": df_products_clean,
    "sellers_cleaned.csv": df_sellers_clean,
    "translation_cleaned.csv": df_translation_clean,
    "online_retail_cleaned.csv": df_online_retail_clean,
    "complaints_cleaned.csv": df_complaints_clean
}

for filename, dataframe in cleaned_datasets.items():
    filepath = os.path.join("data/cleaned", filename)
    dataframe.to_csv(filepath, index=False)
    print(f"Saved: {filename} → {dataframe.shape}")

Saved: customers_cleaned.csv → (99441, 5)
Saved: geolocation_cleaned.csv → (738332, 5)
Saved: order_items_cleaned.csv → (112650, 7)
Saved: payments_cleaned.csv → (103886, 5)
Saved: reviews_cleaned.csv → (99224, 7)
Saved: orders_cleaned.csv → (99441, 8)
Saved: products_cleaned.csv → (32951, 9)
Saved: sellers_cleaned.csv → (3095, 4)
Saved: translation_cleaned.csv → (71, 2)
Saved: online_retail_cleaned.csv → (514912, 8)
Saved: complaints_cleaned.csv → (11, 3)


In [124]:

# Save the final integrated ReturnIQ dataset used for EDA.
# This file contains one row per order item and all engineered features.
integrated_path = os.path.join("data/cleaned", "returniq_integrated_features.csv")
order_items_complete.to_csv(integrated_path, index=False)

print(f"Saved integrated ReturnIQ dataset: {integrated_path}")
print("Shape:", order_items_complete.shape)


Saved integrated ReturnIQ dataset: data/cleaned\returniq_integrated_features.csv
Shape: (112650, 71)


In [125]:
print("\n========== SAVED CLEANED DATASETS ==========")

for filename in cleaned_datasets:
    filepath = os.path.join("data/cleaned", filename)
    print(
        f"{filename}: "
        f"{os.path.exists(filepath)}"
    )


========== SAVED CLEANED DATASETS ==========
customers_cleaned.csv: True
geolocation_cleaned.csv: True
order_items_cleaned.csv: True
payments_cleaned.csv: True
reviews_cleaned.csv: True
orders_cleaned.csv: True
products_cleaned.csv: True
sellers_cleaned.csv: True
translation_cleaned.csv: True
online_retail_cleaned.csv: True
complaints_cleaned.csv: True


## Cleaning Complete

The cleaned datasets and the integrated ReturnIQ analytical dataset are now ready.

Next: **03_Exploratory_Analysis.ipynb**